# Segmentación de imagen con YOLOv8 + Fine-tuning

Este notebook:
1. Instala/carga `ultralytics` (YOLOv8).
2. **Hace fine-tuning** de un modelo de segmentación de YOLOv8 (parte de pesos preentrenados en COCO)
   sobre tu propio dataset, para mejorar la segmentación en tu caso de uso específico.
3. Carga el modelo ya afinado (fine-tuned).
4. El usuario carga una imagen.
5. Corre la segmentación con el modelo afinado y retorna la imagen segmentada.

> **Nota:** el fine-tuning requiere un dataset propio en formato de segmentación de YOLO
> (ver sección 2). Si aún no tienes ese dataset, puedes correr igual el notebook usando
> directamente el modelo preentrenado (salta la sección 3 de entrenamiento) y volver a esta
> parte cuando tengas tus imágenes y anotaciones listas.


## 1. Instalar e importar librerías

Si `ultralytics` no está instalado en el entorno, la siguiente celda lo instala
(solo es necesario ejecutarla una vez).

In [ ]:
import sys
!{sys.executable} -m pip install -q ultralytics


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO


## 2. Preparar el dataset para fine-tuning

Para afinar (fine-tune) el modelo de segmentación necesitas un dataset en **formato YOLO de segmentación**:

```
dataset/
├── images/
│   ├── train/   -> imágenes de entrenamiento (.jpg, .png, ...)
│   └── val/     -> imágenes de validación
├── labels/
│   ├── train/   -> un .txt por imagen, mismo nombre, con los polígonos de cada objeto
│   └── val/
└── data.yaml    -> configuración del dataset (rutas y nombres de clases)
```

Cada línea de un archivo `.txt` en `labels/` representa un objeto segmentado, con el formato:

```
<clase_id> x1 y1 x2 y2 x3 y3 ... xn yn
```

donde `x_i, y_i` son las coordenadas del polígono del contorno del objeto, **normalizadas entre 0 y 1**
(relativas al ancho y alto de la imagen).

El archivo `data.yaml` debe verse así:

```yaml
path: C:/ruta/a/dataset
train: images/train
val: images/val
names:
  0: clase_1
  1: clase_2
  # ... una entrada por cada clase que quieras segmentar
```

> Si no tienes las anotaciones todavía, herramientas como **Roboflow**, **CVAT** o **LabelMe**
> permiten etiquetar imágenes con polígonos y exportar directamente en formato YOLO-seg.

In [ ]:
RUTA_DATA_YAML = "dataset/data.yaml"  # <-- cambia esta ruta por la de tu archivo data.yaml

# Verificación rápida de que el archivo existe antes de entrenar
if os.path.exists(RUTA_DATA_YAML):
    print(f"Archivo de configuración encontrado: {RUTA_DATA_YAML}")
else:
    print(f"ATENCIÓN: no se encontró '{RUTA_DATA_YAML}'. Ajusta la ruta antes de entrenar.")


## 3. Fine-tuning del modelo de segmentación

Se parte de un modelo **preentrenado en COCO** (`yolov8n-seg.pt`) y se continúa el entrenamiento
sobre tu dataset propio (transfer learning), en vez de entrenar desde cero.

Parámetros editables:
- `MODELO_BASE`: tamaño del modelo base (`yolov8n-seg.pt` es el más liviano/rápido;
  `s`, `m`, `l`, `x` son progresivamente más grandes y precisos, pero más lentos de entrenar).
- `EPOCHS`: número de épocas de entrenamiento.
- `IMGSZ`: tamaño de imagen usado durante el entrenamiento.
- `BATCH`: tamaño de lote (redúcelo si tienes poca memoria de GPU/CPU).

In [ ]:
MODELO_BASE = "yolov8n-seg.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = 8
NOMBRE_EXPERIMENTO = "finetune_segmentacion"

modelo_base = YOLO(MODELO_BASE)

resultados_entrenamiento = modelo_base.train(
    data=RUTA_DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name=NOMBRE_EXPERIMENTO
)


## 4. Cargar el modelo ya afinado (fine-tuned)

`ultralytics` guarda automáticamente los pesos del entrenamiento en
`runs/segment/<NOMBRE_EXPERIMENTO>/weights/best.pt` (el mejor checkpoint según la métrica de validación).

Esta celda arma esa ruta y carga el modelo afinado para usarlo en inferencia.

In [ ]:
RUTA_MODELO_AFINADO = os.path.join("runs", "segment", NOMBRE_EXPERIMENTO, "weights", "best.pt")
print(f"Cargando modelo afinado desde: {RUTA_MODELO_AFINADO}")

modelo = YOLO(RUTA_MODELO_AFINADO)


### (Alternativa) Usar directamente un modelo ya afinado previamente

Si ya entrenaste antes y solo quieres cargar un `best.pt` existente (sin volver a entrenar),
comenta la celda de la sección 3 y usa esta en su lugar.

In [ ]:
# RUTA_MODELO_AFINADO = "runs/segment/finetune_segmentacion/weights/best.pt"
# modelo = YOLO(RUTA_MODELO_AFINADO)


## 5. Cargar la imagen del usuario

Modifica `RUTA_IMAGEN` con la ruta de tu imagen (jpg, png, etc.).

In [ ]:
RUTA_IMAGEN = "taller_mecanico.jpg"  # <-- cambia esta ruta por la de tu imagen

def cargar_imagen_rgb(ruta):
    """Carga una imagen desde disco y la devuelve como arreglo numpy en formato RGB (H, W, 3)."""
    img = Image.open(ruta).convert("RGB")
    return np.array(img)

imagen_original = cargar_imagen_rgb(RUTA_IMAGEN)
print(f"Dimensiones de la imagen: {imagen_original.shape}")


## 6. Ejecutar la segmentación con el modelo afinado

In [ ]:
CONFIANZA_MINIMA = 0.25  # umbral de confianza para mostrar una detección

resultados = modelo.predict(source=imagen_original, conf=CONFIANZA_MINIMA, verbose=False)
resultado = resultados[0]

if resultado.masks is None:
    print("No se detectó ningún elemento en la imagen.")
else:
    n_objetos = len(resultado.masks)
    print(f"Elementos segmentados: {n_objetos}")
    for i in range(n_objetos):
        clase_id = int(resultado.boxes.cls[i])
        nombre_clase = modelo.names[clase_id]
        confianza = float(resultado.boxes.conf[i])
        print(f"  {i+1}. {nombre_clase} (confianza: {confianza:.2f})")


## 7. Obtener la imagen segmentada

`resultado.plot()` dibuja las máscaras, cajas y etiquetas sobre la imagen y la devuelve
como un arreglo numpy en formato BGR (convención de OpenCV), por lo que se convierte a RGB.

In [ ]:
imagen_segmentada_bgr = resultado.plot()          # imagen con máscaras/cajas dibujadas (BGR)
imagen_segmentada = imagen_segmentada_bgr[:, :, ::-1]  # convertir BGR -> RGB


## 8. Subplot: imagen original vs. imagen segmentada (modelo afinado)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 7))

ejes[0].imshow(imagen_original)
ejes[0].set_title("Imagen original")
ejes[0].axis("off")

ejes[1].imshow(imagen_segmentada)
ejes[1].set_title("Imagen segmentada (YOLOv8 fine-tuned)")
ejes[1].axis("off")

plt.tight_layout()
plt.show()


## 9. (Opcional) Guardar la imagen segmentada en disco

In [ ]:
RUTA_SALIDA = "imagen_segmentada_finetuned.png"

Image.fromarray(imagen_segmentada).save(RUTA_SALIDA)
print(f"Imagen segmentada guardada en: {RUTA_SALIDA}")
